In [1]:
%pwd

'c:\\Users\\Tina Khatri\\ASTRABOT\\research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\Tina Khatri\\ASTRABOT'

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [5]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


In [6]:
def load_pdf_file(Data):
    loader = DirectoryLoader(Data, glob="*.pdf",loader_cls = PyPDFLoader)
    documents = loader.load()
    return documents

In [7]:
extracted_data = load_pdf_file(r"C:/Users/Tina Khatri/ASTRABOT/Data")




In [8]:
# extracted_data 

In [9]:
#splitting into chunks

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500 , chunk_overlap = 20 )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [10]:
text_chunks = text_split(extracted_data)
print("Length of text chunks:" , len(text_chunks))

Length of text chunks: 5552


In [11]:
# text_chunks
from langchain_community.embeddings import HuggingFaceEmbeddings


In [12]:
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [13]:
embeddings = download_hugging_face_embeddings()

C:\Users\Tina Khatri\AppData\Local\Temp\ipykernel_30628\4189599231.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")


In [14]:
query_result =  embeddings.embed_query("What is ASTRABOT?")
print("Query Result:" , query_result)
print("Length" , len(query_result))

Query Result: [-0.10675754398107529, -0.00220382958650589, -0.15943196415901184, 0.07742665708065033, -0.02380170300602913, -0.055547311902046204, 0.03422800451517105, 0.0645129606127739, -0.013904226943850517, -0.0552971251308918, 0.04986560717225075, 0.025634311139583588, 0.019568150863051414, -0.02661139890551567, -0.04835928604006767, 0.012768229469656944, 0.11616262048482895, -0.0403701588511467, 0.0050050606951117516, -0.0408337339758873, 0.03937433287501335, 0.045785289257764816, 0.060664206743240356, 0.02186109870672226, 0.038599271327257156, 0.03155459091067314, -0.08726698160171509, -0.10786779224872589, 0.033617179840803146, -0.13662727177143097, 0.020033396780490875, -0.02761898934841156, 0.006466743070632219, 0.03169645741581917, 0.08623655140399933, 0.043961457908153534, -0.06374046206474304, -0.0031541220378130674, -0.0052258106879889965, -0.031211435794830322, -0.011129466816782951, -0.020261036232113838, -0.008427117951214314, -0.03356316685676575, -0.0047209900803864,

In [15]:
query_result

[-0.10675754398107529,
 -0.00220382958650589,
 -0.15943196415901184,
 0.07742665708065033,
 -0.02380170300602913,
 -0.055547311902046204,
 0.03422800451517105,
 0.0645129606127739,
 -0.013904226943850517,
 -0.0552971251308918,
 0.04986560717225075,
 0.025634311139583588,
 0.019568150863051414,
 -0.02661139890551567,
 -0.04835928604006767,
 0.012768229469656944,
 0.11616262048482895,
 -0.0403701588511467,
 0.0050050606951117516,
 -0.0408337339758873,
 0.03937433287501335,
 0.045785289257764816,
 0.060664206743240356,
 0.02186109870672226,
 0.038599271327257156,
 0.03155459091067314,
 -0.08726698160171509,
 -0.10786779224872589,
 0.033617179840803146,
 -0.13662727177143097,
 0.020033396780490875,
 -0.02761898934841156,
 0.006466743070632219,
 0.03169645741581917,
 0.08623655140399933,
 0.043961457908153534,
 -0.06374046206474304,
 -0.0031541220378130674,
 -0.0052258106879889965,
 -0.031211435794830322,
 -0.011129466816782951,
 -0.020261036232113838,
 -0.008427117951214314,
 -0.0335631668

In [16]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents = text_chunks,
    embedding = embeddings,
    persist_directory= "./chroma_db"
)
vectorstore.persist()
print(f"stored{len(text_chunks)} vectors")



stored5552 vectors


C:\Users\Tina Khatri\AppData\Local\Temp\ipykernel_30628\864250961.py:8: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [17]:
docstore = Chroma(
    persist_directory = "./chroma_db",
    embedding_function = embeddings
)

C:\Users\Tina Khatri\AppData\Local\Temp\ipykernel_30628\4076156899.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  docstore = Chroma(


In [18]:
rel_docs = docstore.similarity_search("define agriculture", k = 3)
for doc in rel_docs:
    print(f"{doc.page_content[:500]}")

produce more abundantly, and at the same time, to protect it from deterioration and misuse. It is
synonymous with farming–the production of food, fodder and other industrial materials.
B.  Definitions
Agriculture is defined in the Agriculture A ct 1947, as including ‘horticulture, fruit growing, seed
growing, dairy farming and livestock breeding and keeping, the use of land as grazing land, meadow
1.0 AN INTRODUCTION TO AGRICULTURE
A.  Terminology
Agriculture is derived from Latin words Ager and Cultura. Ager means land or field and Cultura means
cultivation. Therefore the term agriculture means cultivation of land. i.e., the science and art of produc-
ing crops and livestock for economic purposes. It is also referred as the science of producing crops and
livestock from the natural resources of the earth. The primary aim of agriculture is to cause the land to
C.  Agriculture as art, science and business of crop production
Agriculture is defined as the art, the science and the business 

In [19]:
##LLM INITIALIZATION
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACE_API")

llm_endpoint = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    temperature=0.1,
    max_new_tokens=512
)

llm = ChatHuggingFace(llm=llm_endpoint)



In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful AI assistant.

You are an expert agricultural advisor for farmers.

Rules:
- Never say "based on the context"
- Never explain the source
- Answer directly
If the answer is not in the context, say "I don't know".
If the client greets you , greet them back politely.

Context:
{context}

Question:
{question}

Answer:
"""
)
retriever = docstore.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [21]:

query = """
Explain in detail the working of the robot we have built.
Cover the following points clearly:
1. Overall purpose of the robot
2. Hardware components (chassis, motors, tank, pumps, sprinklers, sensors if any)
3. Software/control logic
4. Step-by-step working process
5. How it operates in a real agricultural field
6. Advantages of this robot

Write a detailed explanation of at least 250–300 words.
"""
response = rag_chain.invoke(query)
print(response.content)



 The robot we have built is a mobile agricultural robot designed for row-based farming, specifically for selective spraying and monitoring of crops. The overall purpose of this robot is to optimize the use of agricultural resources, reduce the amount of water and chemicals used, and improve crop health and yield.

The hardware components of the robot include a mobile robotic platform, a four-wheel chassis driven by DC motors, a pesticide storage tank, pumping motors, and sprinklers. The mobile platform is designed to move along crop paths, enabling selective spraying and monitoring of crops. The chassis, driven by DC motors, provides controlled movement along the crop paths. The pesticide storage tank, pumping motors, and sprinklers are mounted on the platform to enable selective spraying. A camera module is positioned to capture clear images of plant leaves during movement, and soil sensors are placed close to the ground to measure soil-related parameters such as moisture and conditio